# 🩺 WORKPLACE DEMO TRỰC TIẾP: TẢI SỐ ĐO LÊN APP & KIỂM CHỨNG PHÂN TÍCH DỮ LIỆU (CHƯƠNG 4)
**Đề tài:** Phân tích nghiệp vụ ứng dụng hỗ trợ người thân trong hệ thống AI Care Companion  
**Sinh viên thực hiện:** Nguyễn Phạm Kiều Duyên (MSSV: 221124029111 - Lớp 48K29.1)  
**Link Google Colab:** https://colab.research.google.com/drive/1krKezs51O0TgHA-gpv1tnN_gldZtPRCm  
**Link Web App Người thân:** https://myka-care-companion.pages.dev  

--- 
### Hướng dẫn sử dụng khi báo cáo với Thầy:
1. **Khi Thầy yêu cầu: *“Thử tải số đo huyết áp, nhịp tim 1 ngày lên App cho thầy xem”*** $\rightarrow$ Bấm nút **Play (▶️)** ở **Ô Code số 1** bên dưới. Dữ liệu huyết áp & nhịp tim 1 ngày (khung sáng 09:00 và khung tối 19:00) sẽ được bắn trực tiếp qua API lên App. Mở trang Web App sẽ thấy ngay số đo mới trên Biểu đồ và Lịch sử đo.
2. **Khi Thầy yêu cầu: *“Kiểm chứng ma trận phân loại 50 câu kiểm tra độc lập”*** $\rightarrow$ Bấm nút **Play (▶️)** ở **Ô Code số 2** bên dưới. Thuật toán sẽ chạy trực tiếp trên 50 câu kiểm tra độc lập (`B-041` đến `B-090`) và in ra đúng Ma trận nhầm lẫn $4 \times 4$ (**43/50 câu đúng = 86,0%**, **Macro-F1 = 84,2%**).
3. **Khi Thầy yêu cầu: *“Xem đồng bộ lịch uống thuốc, nhật ký trò chuyện và link Chat Group Telegram”*** $\rightarrow$ Bấm nút **Play (▶️)** ở **Ô Code số 3**.

In [ ]:
# ==============================================================================
# DEMO TẢI SỐ ĐO HUYẾT ÁP & NHỊP TIM 1 NGÀY (SÁNG 09:00 & TỐI 19:00) LÊN APP
# Gửi trực tiếp lên cả Backend API (103.147.186.177:5004) và App Web Public
# ==============================================================================
import json
import urllib.request
from datetime import datetime, timezone, timedelta

VN_TZ = timezone(timedelta(hours=7))
today_vn = datetime.now(VN_TZ).strftime('%Y-%m-%d')

# Cấu hình 2 đích nhận dữ liệu:
# 1. Link Web App Public của bạn trên Cloudflare Pages
PUBLIC_APP_API = "https://myka-care-companion.pages.dev"
# 2. Máy chủ REST API Robot Myka của anh Thế Khanh (API_INTEGRATION_GUIDE.md)
ROBOT_BACKEND_API = "http://103.147.186.177:5004"

# Hai lượt đo huyết áp & nhịp tim trong 1 ngày để demo cho Thầy xem
one_day_measurements = [
    {
        "slot_name": "Buổi sáng (09:00)",
        "sys": 121,
        "dia": 78,
        "pul": 72,
        "ihb": False,
        "robot_id": "ROBOT-MYKA-001",
        "note": f"APP-SYNC-DEMO: Số đo Robot Myka khung sáng 09:00 ngày {today_vn}"
    },
    {
        "slot_name": "Buổi tối (19:00)",
        "sys": 126,
        "dia": 81,
        "pul": 75,
        "ihb": False,
        "robot_id": "ROBOT-MYKA-001",
        "note": f"APP-SYNC-DEMO: Số đo Robot Myka khung tối 19:00 ngày {today_vn}"
    }
]

def post_json(url, payload, origin_header=None):
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if origin_header:
        headers["Origin"] = origin_header
    req = urllib.request.Request(url, data=json.dumps(payload).encode("utf-8"), headers=headers, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            return resp.status, json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        return 0, {"status": "offline_or_skipped", "detail": str(e)}

def get_json(url):
    req = urllib.request.Request(url, headers={"Accept": "application/json"}, method="GET")
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            return resp.status, json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        return 0, {"status": "offline_or_skipped", "detail": str(e)}

print(f"=== BẮT ĐẦU TẢI SỐ ĐO HUYẾT ÁP & NHỊP TIM NGÀY {today_vn} LÊN APP ===")
for m in one_day_measurements:
    print(f"\n>>> Đang tải {m['slot_name']}: Huyết áp {m['sys']}/{m['dia']} mmHg | Nhịp tim {m['pul']} lần/phút")
    
    # 1. Gửi lên Backend API theo chuẩn POST /api/bpm (API_INTEGRATION_GUIDE.md)
    st_backend, res_backend = post_json(f"{ROBOT_BACKEND_API}/api/bpm", {
        "sys": m["sys"], "dia": m["dia"], "pul": m["pul"], "ihb": m["ihb"], "robot_id": m["robot_id"]
    })
    print(f"  [1] Phản hồi từ Robot Backend ({ROBOT_BACKEND_API}/api/bpm) [HTTP {st_backend}]:")
    print("     ", json.dumps(res_backend, ensure_ascii=False))
    
    # 2. Đồng bộ trực tiếp lên Database D1 của Link Web App Public (https://myka-care-companion.pages.dev)
    st_app, res_app = post_json(
        f"{PUBLIC_APP_API}/api/device/bp-record/E001",
        {"systolic": m["sys"], "diastolic": m["dia"], "heart_rate": m["pul"], "note": m["note"]},
        origin_header=PUBLIC_APP_API
    )
    print(f"  [2] Phản hồi từ Link Web App ({PUBLIC_APP_API}) [HTTP {st_app}]:")
    print("     ", json.dumps(res_app, ensure_ascii=False)[:300])

# Kiểm tra đọc lại dữ liệu trên App sau khi tải lên
print("\n=== ĐỌC LẠI DỮ LIỆU BIỂU ĐỒ 7 NGÀY TRÊN APP NGƯỜI THÂN (GET /api/family/health-summary/E001?days=7) ===")
st_sum, sum_data = get_json(f"{PUBLIC_APP_API}/api/family/health-summary/E001?days=7")
readings = (sum_data.get("data") or sum_data).get("daily_readings", [])
print(f"Tổng số bản ghi hiển thị trên biểu đồ 7 ngày của App: {len(readings)} bản ghi")
for r in readings[:5]:
    print(f"  • Mã: {r.get('id')} | Huyết áp: {r.get('systolic')}/{r.get('diastolic')} mmHg | Nhịp tim: {r.get('heart_rate')} bpm | Giờ đo: {r.get('timestamp')}")
print("\n👉 Bây giờ bạn mở/tải lại trang https://myka-care-companion.pages.dev là thấy ngay số đo mới trên Biểu đồ và Bảng lịch sử!")


## 2. KIỂM CHỨNG TRỰC TIẾP MA TRẬN PHÂN LOẠI 50 CÂU KIỂM TRA ĐỘC LẬP (`split=test` - MỤC 4.5)
Bấm chạy ô dưới đây để thực thi bộ quy tắc từ khóa & xử lý phủ định trên toàn bộ **50 câu kiểm tra độc lập** (`B-041` đến `B-090`), xuất Ma trận phân loại $4 \times 4$ (Bảng 4.33) và bảng Precision / Recall / F1 theo từng nhãn (Bảng 4.34).

In [ ]:
# ==============================================================================
# BỘ NGỮ LIỆU 90 CÂU NÓI TIẾNG VIỆT (40 CÂU DEV & 50 CÂU HELD-OUT TEST ĐỘC LẬP)
# Do sinh viên Nguyễn Phạm Kiều Duyên xây dựng và rà soát nhãn thủ công 100%
# ==============================================================================
import re
import unicodedata
import numpy as np
import pandas as pd

DEV_40 = [
  [
    "B-001",
    "Hôm nay tôi thấy vui lắm con ạ",
    "pos",
    "plain",
    "Nêu rõ cảm xúc vui của người nói."
  ],
  [
    "B-002",
    "Tôi khỏe, ăn ngon ngủ tốt",
    "pos",
    "plain",
    "Tự mô tả khỏe, ăn ngon và ngủ tốt; sắc thái tích cực về thể trạng, không phải đo sức khỏe."
  ],
  [
    "B-003",
    "Sáng nay tôi thấy thoải mái",
    "pos",
    "plain",
    "Tự mô tả cảm giác thoải mái."
  ],
  [
    "B-004",
    "Con cháu về thăm, bà hạnh phúc lắm",
    "pos",
    "plain",
    "Nêu trực tiếp hạnh phúc khi có con cháu."
  ],
  [
    "B-005",
    "Hôm nay trời đẹp, tôi vui vẻ",
    "pos",
    "plain",
    "Có từ vui vẻ mô tả người nói, không chỉ nhận xét thời tiết."
  ],
  [
    "B-006",
    "Tôi thấy ổn, không có gì phải lo",
    "pos",
    "negation",
    "Tự thấy ổn và không lo; có khẳng định tích cực ngoài phủ định."
  ],
  [
    "B-007",
    "Tôi buồn quá",
    "neg",
    "plain",
    "Bộc lộ buồn trực tiếp."
  ],
  [
    "B-008",
    "Hôm nay mệt lắm, không muốn làm gì",
    "neg",
    "plain",
    "Tự mô tả mệt và không muốn hoạt động; sắc thái khó chịu về bản thân."
  ],
  [
    "B-009",
    "Tôi thấy cô đơn",
    "neg",
    "plain",
    "Nêu cảm giác cô đơn."
  ],
  [
    "B-010",
    "Đêm qua tôi mất ngủ, người khó chịu",
    "neg",
    "plain",
    "Mất ngủ đi cùng tự mô tả khó chịu; chỉ nhãn sắc thái, không chẩn đoán."
  ],
  [
    "B-011",
    "Tôi đau lưng, chán lắm",
    "neg",
    "plain",
    "Có cảm xúc chán đi kèm đau lưng."
  ],
  [
    "B-012",
    "Hôm nay tôi thấy lo lắng",
    "neg",
    "plain",
    "Nêu lo lắng trực tiếp."
  ],
  [
    "B-013",
    "Tôi chóng mặt, thấy tệ",
    "neg",
    "plain",
    "Tự nói thấy tệ đi kèm chóng mặt."
  ],
  [
    "B-014",
    "Cũng bình thường thôi",
    "neu",
    "plain",
    "Tự mô tả bình thường."
  ],
  [
    "B-015",
    "Hôm nay tạm được",
    "neu",
    "plain",
    "Tạm được là trạng thái chấp nhận được, không tích cực rõ."
  ],
  [
    "B-016",
    "Không vui không buồn",
    "neu",
    "negation",
    "Phủ định đồng thời vui và buồn; giữ trung tính."
  ],
  [
    "B-017",
    "Cũng được, không có gì đặc biệt",
    "neu",
    "plain",
    "Cũng được và không đặc biệt thể hiện trung tính."
  ],
  [
    "B-018",
    "Tôi không vui",
    "neg",
    "negation",
    "Không vui là phủ định trạng thái tích cực; sắc thái tiêu cực trong bộ quy ước."
  ],
  [
    "B-019",
    "Hôm nay tôi chẳng khỏe tí nào",
    "neg",
    "negation",
    "Chẳng khỏe thể hiện cảm nhận thể trạng không tốt; không suy ra bệnh."
  ],
  [
    "B-020",
    "Tôi không buồn đâu con",
    "neu",
    "negation",
    "Không buồn không đủ để suy thành vui; giữ trung tính."
  ],
  [
    "B-021",
    "Hôm nay không mệt",
    "neu",
    "negation",
    "Không mệt là phủ định khó chịu, chưa khẳng định cảm xúc tích cực."
  ],
  [
    "B-022",
    "Chẳng thấy đau gì cả",
    "neu",
    "negation",
    "Không đau không đồng nghĩa vui hoặc khỏe toàn diện; trung tính theo quy ước."
  ],
  [
    "B-023",
    "toi buon qua",
    "neg",
    "no_diacritics",
    "Câu không dấu tương đương tôi buồn quá."
  ],
  [
    "B-024",
    "hom nay khoe lam",
    "pos",
    "no_diacritics",
    "Câu không dấu tự nói khỏe lắm."
  ],
  [
    "B-025",
    "toi met lam khong muon an",
    "neg",
    "no_diacritics",
    "Câu không dấu tự nói mệt, không muốn ăn."
  ],
  [
    "B-026",
    "binh thuong thoi",
    "neu",
    "no_diacritics",
    "Câu không dấu mô tả bình thường."
  ],
  [
    "B-027",
    "Mấy giờ rồi con",
    "none",
    "no_keyword",
    "Hỏi giờ, không cung cấp trạng thái bản thân."
  ],
  [
    "B-028",
    "Nhắc tôi uống thuốc nhé",
    "none",
    "no_keyword",
    "Yêu cầu nhắc thuốc không xác nhận đã uống và không cho biết tâm trạng."
  ],
  [
    "B-029",
    "Hôm nay trời mưa",
    "none",
    "no_keyword",
    "Nhận xét thời tiết, không cho biết cảm xúc người nói."
  ],
  [
    "B-030",
    "Con gọi cho bà nhé",
    "none",
    "no_keyword",
    "Yêu cầu gọi điện không thể tự suy ra cô đơn."
  ],
  [
    "B-031",
    "Cho tôi nghe nhạc đi",
    "none",
    "no_keyword",
    "Yêu cầu nghe nhạc không đủ suy ra cảm xúc."
  ],
  [
    "B-032",
    "toi muon xem tivi",
    "none",
    "no_keyword",
    "Muốn xem tivi không cho biết tâm trạng."
  ],
  [
    "B-033",
    "Tôi hơi mệt",
    "neg",
    "intensity",
    "Có mệt nhẹ; mức nhẹ không đảo chiều sắc thái."
  ],
  [
    "B-034",
    "Tôi rất vui",
    "pos",
    "intensity",
    "Khẳng định rất vui."
  ],
  [
    "B-035",
    "Mệt nhưng vui",
    "neu",
    "mixed",
    "Có đồng thời mệt và vui; dùng nhãn trung tính hỗn hợp theo cùng quy tắc cho mọi câu."
  ],
  [
    "B-036",
    "Hôm nay đau nhiều nhưng tôi vẫn ổn",
    "neu",
    "mixed",
    "Đau nhiều nhưng vẫn ổn: tín hiệu hỗn hợp, không suy ra cảm xúc thuần tích cực."
  ],
  [
    "B-037",
    "Khỏe như vâm",
    "pos",
    "idiom",
    "Thành ngữ khỏe như vâm có sắc thái thể trạng tích cực."
  ],
  [
    "B-038",
    "Vui gớm, đau cả đêm không ngủ được",
    "neg",
    "sarcasm",
    "Vui gớm đi kèm đau suốt đêm là cách nói mỉa mai trong câu mẫu; phụ thuộc ngữ cảnh."
  ],
  [
    "B-039",
    "Hôm nay chán ơi là chán",
    "neg",
    "plain",
    "Lặp từ chán thể hiện sắc thái tiêu cực."
  ],
  [
    "B-040",
    "Tôi thấy yên tâm khi có robot bên cạnh",
    "pos",
    "plain",
    "Khẳng định yên tâm khi có robot."
  ]
]

TEST_50 = [
  [
    "B-041",
    "Sáng nay tôi dậy thấy khỏe hẳn ra",
    "pos",
    "plain",
    "Tự cảm nhận khỏe hơn khi thức dậy."
  ],
  [
    "B-042",
    "Hôm nay tôi vui vì có người đến chơi",
    "pos",
    "plain",
    "Nêu vui do có người đến chơi."
  ],
  [
    "B-043",
    "Ăn xong tôi thấy thoải mái dễ chịu",
    "pos",
    "plain",
    "Khẳng định thoải mái dễ chịu sau ăn."
  ],
  [
    "B-044",
    "Tôi thấy rất hạnh phúc bên các cháu",
    "pos",
    "plain",
    "Nêu hạnh phúc rõ ràng."
  ],
  [
    "B-045",
    "Hôm nay tôi ổn, đi dạo về còn khỏe nữa",
    "pos",
    "plain",
    "Có cả ổn và khỏe trong tự mô tả."
  ],
  [
    "B-046",
    "Tôi thấy yên tâm lắm",
    "pos",
    "plain",
    "Nêu yên tâm trực tiếp."
  ],
  [
    "B-047",
    "Nay tôi tươi tỉnh lắm",
    "pos",
    "out_of_vocab",
    "Tươi tỉnh là mô tả tích cực dù ngoài từ điển hiện tại."
  ],
  [
    "B-048",
    "Tôi thích bài hát này, nghe vui tai",
    "pos",
    "plain",
    "Thích và vui tai là phản ứng tích cực với bài hát; chỉ là sắc thái phát ngôn, chưa đại diện trạng thái cả ngày."
  ],
  [
    "B-049",
    "Con gái gọi điện làm tôi vui cả ngày",
    "pos",
    "plain",
    "Nêu vui cả ngày do cuộc gọi."
  ],
  [
    "B-050",
    "Hôm nay người khỏe, ăn được hai bát cơm",
    "pos",
    "plain",
    "Tự nói khỏe, ăn được; nhãn cảm nhận thể trạng, không chứng minh dinh dưỡng đạt."
  ],
  [
    "B-051",
    "Mừng quá, hôm nay có tin tốt",
    "pos",
    "out_of_vocab",
    "Mừng là cảm xúc tích cực trực tiếp."
  ],
  [
    "B-052",
    "toi vui lam, khoe nua",
    "pos",
    "no_diacritics",
    "Câu không dấu có vui và khỏe."
  ],
  [
    "B-053",
    "Tôi buồn lắm, chẳng muốn nói chuyện",
    "neg",
    "plain",
    "Nêu buồn và không muốn nói chuyện."
  ],
  [
    "B-054",
    "Hôm nay tôi mệt, đau khắp người",
    "neg",
    "plain",
    "Mệt và đau là tự mô tả khó chịu; không suy ra chẩn đoán tâm thần."
  ],
  [
    "B-055",
    "Tôi sợ ở nhà một mình",
    "neg",
    "plain",
    "Nêu sợ ở một mình."
  ],
  [
    "B-056",
    "Trời lạnh làm tôi khó chịu",
    "neg",
    "plain",
    "Tự nói khó chịu do lạnh; không chỉ nhận xét thời tiết."
  ],
  [
    "B-057",
    "Tối qua tôi mất ngủ cả đêm",
    "neg",
    "plain",
    "Tự báo mất ngủ: sắc thái vấn đề thể trạng, không đồng nghĩa cảm xúc buồn."
  ],
  [
    "B-058",
    "Tôi thấy chán nản",
    "neg",
    "plain",
    "Nêu chán nản."
  ],
  [
    "B-059",
    "Người tôi uể oải, nặng nề",
    "neg",
    "out_of_vocab",
    "Uể oải nặng nề là tự mô tả thể trạng không dễ chịu."
  ],
  [
    "B-060",
    "Tôi nhớ ông nhà tôi, buồn lắm",
    "neg",
    "plain",
    "Nhớ người thân đi kèm cảm xúc buồn rõ."
  ],
  [
    "B-061",
    "Tôi lo lắm, tim đập nhanh",
    "neg",
    "plain",
    "Nêu lo; tim đập nhanh không được coi là số đo cảm biến."
  ],
  [
    "B-062",
    "toi dau dau va chong mat",
    "neg",
    "no_diacritics",
    "Câu không dấu tự báo đau đầu và chóng mặt; nhãn khó chịu về thể trạng."
  ],
  [
    "B-063",
    "Hôm nay tôi thấy tệ lắm",
    "neg",
    "plain",
    "Nêu cảm giác tệ trực tiếp."
  ],
  [
    "B-064",
    "Tôi chán, không muốn ăn gì cả",
    "neg",
    "plain",
    "Chán và không muốn ăn có sắc thái tiêu cực."
  ],
  [
    "B-065",
    "Tôi bực mình vì cái chân đau",
    "neg",
    "plain",
    "Bực mình là cảm xúc tiêu cực trực tiếp."
  ],
  [
    "B-066",
    "Cô đơn quá, chẳng ai nói chuyện với tôi",
    "neg",
    "plain",
    "Cô đơn là cảm xúc tiêu cực; không có người trò chuyện là ngữ cảnh."
  ],
  [
    "B-067",
    "Bình thường, chẳng có gì mới",
    "neu",
    "plain",
    "Tự mô tả bình thường, không có gì mới."
  ],
  [
    "B-068",
    "Cũng tạm ổn thôi",
    "neu",
    "plain",
    "Tạm ổn không phải vui rõ ràng; giữ trung tính."
  ],
  [
    "B-069",
    "Hôm nay cũng như mọi ngày",
    "none",
    "out_of_vocab",
    "Như mọi ngày chỉ là so sánh với nền chưa biết; chưa đủ dữ liệu phân loại trạng thái."
  ],
  [
    "B-070",
    "Không buồn, cũng không vui lắm",
    "neu",
    "negation",
    "Phủ định buồn và giảm mức vui; nhãn trung tính, có mơ hồ ngữ cảnh."
  ],
  [
    "B-071",
    "Tôi không mệt lắm",
    "neu",
    "negation",
    "Không mệt lắm có thể là phủ định hoặc mệt nhẹ; giữ trung tính nhưng đánh dấu mơ hồ."
  ],
  [
    "B-072",
    "Chẳng có gì phải lo",
    "neu",
    "negation",
    "Không có gì phải lo chỉ phủ định lo, chưa khẳng định vui."
  ],
  [
    "B-073",
    "Sáng nay bình thường thôi con",
    "neu",
    "plain",
    "Tự mô tả bình thường."
  ],
  [
    "B-074",
    "Không có gì đặc biệt hôm nay",
    "none",
    "plain",
    "Không có gì đặc biệt không nêu cảm giác; chưa đủ dữ liệu nếu đứng riêng."
  ],
  [
    "B-075",
    "Hôm nay thứ mấy vậy con",
    "none",
    "no_keyword",
    "Hỏi thứ trong tuần không cho biết tâm trạng."
  ],
  [
    "B-076",
    "Bật đài cho tôi nhé",
    "none",
    "no_keyword",
    "Yêu cầu bật đài không cho biết tâm trạng."
  ],
  [
    "B-077",
    "Ngày mai con có đến không",
    "none",
    "no_keyword",
    "Hỏi lịch đến thăm không đủ suy ra cô đơn hoặc buồn."
  ],
  [
    "B-078",
    "Nhắc tôi tái khám tuần sau",
    "none",
    "no_keyword",
    "Yêu cầu nhắc tái khám không cho biết cảm xúc."
  ],
  [
    "B-079",
    "Cho tôi hỏi giờ ăn cơm",
    "none",
    "no_keyword",
    "Hỏi giờ ăn không cho biết tâm trạng."
  ],
  [
    "B-080",
    "Tôi cần uống nước",
    "none",
    "no_keyword",
    "Nhu cầu uống nước không cho biết tâm trạng hoặc mức mất nước."
  ],
  [
    "B-081",
    "toi muon goi cho con",
    "none",
    "no_keyword",
    "Yêu cầu gọi con không đủ suy ra trạng thái cảm xúc."
  ],
  [
    "B-082",
    "Tivi hỏng rồi con ơi",
    "none",
    "no_keyword",
    "Tivi hỏng là sự kiện; chưa nói người nói buồn hoặc bực."
  ],
  [
    "B-083",
    "Vui thì vui nhưng tôi vẫn đau lưng",
    "neu",
    "mixed",
    "Có vui và đau lưng; nhãn hỗn hợp theo cùng quy tắc với câu 35."
  ],
  [
    "B-084",
    "Mệt nhưng tôi thấy vui vì cháu về",
    "neu",
    "mixed",
    "Có mệt và vui; đổi sang trung tính hỗn hợp để nhất quán với câu 35, không ưu tiên chỉ từ vui."
  ],
  [
    "B-085",
    "Khỏe mà cứ buồn buồn thế nào ấy",
    "neu",
    "mixed",
    "Có khỏe và buồn; tín hiệu hỗn hợp."
  ],
  [
    "B-086",
    "Vui gì mà vui, đau cả người",
    "neg",
    "sarcasm",
    "Vui gì mà vui phủ định tích cực, đi kèm đau người; câu mẫu sắc thái tiêu cực."
  ],
  [
    "B-087",
    "Chả vui gì cả",
    "neg",
    "negation",
    "Chả vui gì cả phủ định cảm xúc vui."
  ],
  [
    "B-088",
    "Không khỏe, tôi muốn nằm nghỉ",
    "neg",
    "negation",
    "Không khỏe và muốn nghỉ; sắc thái thể trạng tiêu cực."
  ],
  [
    "B-089",
    "Chưa thấy khỏe hẳn",
    "neg",
    "negation",
    "Chưa khỏe hẳn nêu chưa phục hồi hoàn toàn; sắc thái khó chịu nhẹ, có mơ hồ."
  ],
  [
    "B-090",
    "Ổn rồi, không đau nữa",
    "pos",
    "negation",
    "Khẳng định ổn rồi cùng không đau nữa; có tín hiệu tích cực độc lập."
  ]
]

# Từ điển trọng số cảm xúc (xây dựng trên tập DEV_40, đóng băng trước khi chạy TEST_50)
LEX = {
    "vui vẻ": 2, "vui": 2, "khỏe": 2, "hạnh phúc": 3, "thoải mái": 2, "ổn": 1, "tốt": 1,
    "yên tâm": 1, "thích": 1, "ngon": 1,
    "buồn": -2, "mệt": -2, "chán": -2, "đau": -2, "lo lắng": -2, "lo": -1, "cô đơn": -3,
    "khó chịu": -2, "mất ngủ": -2, "tệ": -2, "sợ": -2, "chóng mặt": -2, "bực": -1,
    "bình thường": 0, "tạm được": 0, "cũng được": 0,
}
NEGATORS = {"không", "chẳng", "chả", "chưa"}
BOOST_AFTER = {"lắm", "quá"}
BOOST_BEFORE = {"rất": 1.5}
SOFT_BEFORE = {"hơi": 0.5}
NEG_NEG_FACTOR = 0.4
NEG_WINDOW = 4

def strip_acc(s):
    s = s.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[.!?;:]", " ", text)
    text = text.replace(",", " , ")
    return text.split()

def build_lex(no_diacritics):
    if not no_diacritics:
        return LEX, NEGATORS
    return {strip_acc(k): v for k, v in LEX.items()}, {strip_acc(n) for n in NEGATORS if n != "chả"}

def predict_sentiment(text):
    nd = strip_acc(text) == text
    lex, negs = build_lex(nd)
    boost_after = {strip_acc(x) if nd else x for x in BOOST_AFTER}
    boost_before = {(strip_acc(k) if nd else k): v for k, v in BOOST_BEFORE.items()}
    soft_before = {(strip_acc(k) if nd else k): v for k, v in SOFT_BEFORE.items()}
    toks = tokenize(text)
    phrases = sorted(lex, key=lambda p: -len(p.split()))
    i, hits, total = 0, [], 0.0
    while i < len(toks):
        matched = None
        for ph in phrases:
            n = len(ph.split())
            if " ".join(toks[i:i + n]) == ph:
                matched = (ph, n)
                break
        if not matched:
            i += 1
            continue
        ph, n = matched
        w = lex[ph]
        negated = False
        j = i - 1
        steps = 0
        while j >= 0 and steps < NEG_WINDOW and toks[j] != ",":
            if toks[j] in negs:
                negated = True
                break
            j -= 1
            steps += 1
        if negated:
            val = -w if w > 0 else (NEG_NEG_FACTOR * abs(w) if w < 0 else 0)
        else:
            val = float(w)
            if i > 0 and toks[i - 1] in boost_before:
                val *= boost_before[toks[i - 1]]
            if i > 0 and toks[i - 1] in soft_before:
                val *= soft_before[toks[i - 1]]
            if i + n < len(toks) and toks[i + n] in boost_after:
                val *= 1.5
        hits.append((ph, round(val, 2), negated))
        total += val
        i += n
    if not hits:
        return "none", 0.0, "Không có từ khóa cảm xúc"
    label = "pos" if total >= 1 else "neg" if total <= -1 else "neu"
    ev = "; ".join(f"{p}({v:+g}{', phủ định' if ng else ''})" for p, v, ng in hits)
    return label, round(total, 2), ev

# Chạy kiểm chứng thực tế trên 50 câu kiểm tra độc lập (TEST_50)
label_order = ["pos", "neu", "neg", "none"]
label_names = {
    "pos": "Tích cực (pos)",
    "neu": "Trung tính/hỗn hợp (neu)",
    "neg": "Tiêu cực (neg)",
    "none": "Chưa đủ dữ liệu (none)"
}

cm = {a: {b: 0 for b in label_order} for a in label_order}
eval_records = []
for cid, text, true_lbl, cat, reason in TEST_50:
    pred_lbl, score_sum, evidence = predict_sentiment(text)
    cm[true_lbl][pred_lbl] += 1
    eval_records.append({
        "Mã câu": cid,
        "Câu nói tiếng Việt": text,
        "Nhóm": cat,
        "Nhãn chuẩn (Ground Truth)": label_names[true_lbl],
        "Kết quả dự đoán": label_names[pred_lbl],
        "Bằng chứng từ khóa": evidence,
        "Đúng/Sai": "ĐÚNG" if true_lbl == pred_lbl else "SAI"
    })

df_predictions = pd.DataFrame(eval_records)
correct_count = sum(cm[a][a] for a in label_order)
total_count = len(TEST_50)
acc = correct_count / total_count * 100

print("==========================================================================")
print(f"KẾT QUẢ KIỂM CHỨNG TRÊN 50 CÂU KIỂM TRA ĐỘC LẬP (HELD-OUT TEST SET)")
print(f"Tổng số câu đúng: {correct_count}/{total_count} câu -> Accuracy = {acc:.1f}%")
print("==========================================================================\n")

# In Ma trận nhầm lẫn 4x4 (Bảng 4.33)
cm_df = pd.DataFrame(
    [[cm[r][c] for c in label_order] for r in label_order],
    index=[f"Chuẩn: {label_names[r]}" for r in label_order],
    columns=[f"Dự đoán: {label_names[c]}" for c in label_order]
)
print("--- BẢNG 4.33: MA TRẬN PHÂN LOẠI TRÊN TOÀN BỘ 50 CÂU KIỂM TRA ---")
print(cm_df.to_string())

# In Precision, Recall, F1 theo từng nhãn (Bảng 4.34)
metrics_rows = []
f1_scores = []
for a in label_order:
    tp = cm[a][a]
    fp = sum(cm[b][a] for b in label_order if b != a)
    fn = sum(cm[a][b] for b in label_order if b != a)
    prec = tp / (tp + fp) * 100 if (tp + fp) else 0
    rec = tp / (tp + fn) * 100 if (tp + fn) else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    f1_scores.append(f1)
    metrics_rows.append({
        "Nhãn": label_names[a],
        "Số câu thật": tp + fn,
        "Đúng (TP)": tp,
        "Precision (%)": f"{prec:.1f}%",
        "Recall (%)": f"{rec:.1f}%",
        "F1-score (%)": f"{f1:.1f}%"
    })

print(f"\n--- BẢNG 4.34: THỐNG KÊ KẾT QUẢ THEO TỪNG NHÃN (Macro-F1 = {np.mean(f1_scores):.1f}%) ---")
print(pd.DataFrame(metrics_rows).to_string(index=False))

print("\n--- CHI TIẾT 50 CÂU KIỂM TRA ĐỘC LẬP (15 CÂU ĐẦU & CÁC CÂU SAI ĐỂ PHÂN TÍCH LỖI) ---")
print(df_predictions.head(15).to_string(index=False))
print("\n--- 7 CÂU PHÂN LOẠI CHƯA KHỚP TRONG 50 CÂU TEST (PHỤC VỤ PHÂN TÍCH LỖI BẢNG 4.35) ---")
print(df_predictions[df_predictions["Đúng/Sai"] == "SAI"].to_string(index=False))


## 3. DEMO ĐỒNG BỘ LỊCH UỐNG THUỐC / KHÁM BỆNH & BẢN TIN NHẬT KÝ TRÒ CHUYỆN + CHAT GROUP TELEGRAM
> **Mục đích:** Chứng minh luồng dữ liệu 2 chiều:
> 1. Người thân thêm lịch uống thuốc trên App $\rightarrow$ gửi qua API đồng bộ xuống Robot Myka.
> 2. Đọc bản tin nhật ký trò chuyện mới nhất hôm nay giữa Ngoại và Robot Myka (nhắc thuốc, hỏi thăm sinh hoạt, đo huyết áp).
> 3. Cung cấp đường dẫn mở trực tiếp **Kênh Chat Group Telegram** (`https://web.telegram.org/a/#-1004358354552`) để người thân tương tác và hỏi thăm tình hình của Ngoại.

In [ ]:
# Thêm lịch uống thuốc mới lên App và kiểm tra Nhật ký trò chuyện + Chat Group Telegram
new_schedule = {
    'type': 'medication',
    'title': 'Uống thuốc huyết áp Amlodipine 5mg',
    'detail': '1 viên sau bữa sáng - Đồng bộ nhắc qua Robot Myka',
    'scheduled_at': f'{today_vn}T08:00:00+07:00',
    'days_of_week': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
}
st_sc, res_sc = post_json(f'{PUBLIC_APP_API}/api/family/schedules/E001', new_schedule, origin_header=PUBLIC_APP_API)
print(f'=== 1. THÊM LỊCH UỐNG THUỐC LÊN APP [HTTP {st_sc}] ===')
print(json.dumps(res_sc, ensure_ascii=False, indent=2))

# Kiểm tra Nhật ký hội thoại thực tế trong ngày hôm nay
st_cv, res_cv = get_json(f'{PUBLIC_APP_API}/api/family/conversation-history/E001?date={today_vn}')
print(f'\n=== 2. BẢN TIN NHẬT KÝ HỘI THOẠI HÔM NAY ({today_vn}) CỦA NGOẠI VỚI ROBOT MYKA [HTTP {st_cv}] ===')
if isinstance(res_cv, dict) and 'messages' in res_cv:
    for msg in res_cv['messages']:
        speaker_name = '🤖 Robot Myka' if msg.get('speaker') == 'myka' else '👵 Ngoại (Bà Hoa)'
        time_str = msg.get('timestamp', '')[11:16]
        print(f'[{time_str}] {speaker_name}: {msg.get("message_text")}')
else:
    print(json.dumps(res_cv, ensure_ascii=False, indent=2))

# Kiểm tra Kênh Chat Telegram kết nối trực tiếp để người thân hỏi Myka
st_tg, res_tg = get_json(f'{PUBLIC_APP_API}/api/family/telegram-consultation/E001')
print(f'\n=== 3. ĐƯỜNG DẪN CHAT GROUP TELEGRAM ĐỂ NGƯỜI THÂN HỎI LẠI MYKA [HTTP {st_tg}] ===')
print(json.dumps(res_tg, ensure_ascii=False, indent=2))
print('\n👉 Link mở Chat Group Telegram trực tiếp trên trình duyệt Web:')
print('   https://web.telegram.org/a/#-1004358354552')
